# Survivor Lifetime Breakdown by Triage

Loads a single-run `patient_log.xlsx`, selects **survivors who needed DCS**
(prescription contains `dcs`), and plots one stacked horizontal bar per
triage class (T1, T2, T3) showing the mean time spent in each phase:

| Phase | Log pattern |
|---|---|
| **Wait transport @ CCP** | `WAITING_TRANSPORT:ccp` → `CLOCK_PAUSED:transport` |
| **Transit CCP → R1** | `CLOCK_PAUSED:transport` → `WAITING_CARE:dcr/pec` |
| **Wait care @ R1** | `WAITING_CARE:dcr/pec` → `CLOCK_PAUSED:care` |
| **In care @ R1** | `CLOCK_PAUSED:care` → `CARE_COMPLETED:*` |
| **Wait transport @ R1** | `WAITING_TRANSPORT:role1` → `CLOCK_PAUSED:transport` |
| **Transit R1 → R2** | `CLOCK_PAUSED:transport` → `WAITING_CARE:dcs` |
| **Wait care @ R2** | `WAITING_CARE:dcs` → `CLOCK_PAUSED:care` |
| **In DCS care** | `CLOCK_PAUSED:care` → `CARE_COMPLETED:*` |

In [ ]:
import sys
from pathlib import Path
_root = Path(".").resolve()
while not (_root / "paths.py").exists():
    _root = _root.parent
sys.path.insert(0, str(_root))
import paths

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

run_folder = paths.SINGLE_RUN_OUTPUT / FOLDER_NAME

pl = pd.read_excel(run_folder / "patient_log.xlsx")
print(f"Loaded {len(pl)} log rows, {pl['patient_id'].nunique()} patients")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# FILTER: DCS survivors only
# ═══════════════════════════════════════════════════════════════════════════

safe_ids = set(pl.loc[pl["action"] == "BECAME_SAFE", "patient_id"].unique())
dcs_ids  = set(pl.loc[pl["diagnosis"].str.contains("dcs", na=False), "patient_id"].unique())
target_ids = safe_ids & dcs_ids

print(f"Survivors: {len(safe_ids)}  |  DCS pathway: {len(dcs_ids)}  |  "
      f"DCS survivors: {len(target_ids)}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# COMPUTE PHASE DURATIONS PER PATIENT
# ═══════════════════════════════════════════════════════════════════════════
# Walk each patient's timeline and measure the 8 phases.

PHASES = [
    "Wait transport @ CCP",
    "Transit CCP → R1",
    "Wait care @ R1",
    "In care @ R1",
    "Wait transport @ R1",
    "Transit R1 → R2",
    "Wait care @ R2",
    "In DCS care",
]

def _next_action(actions, times, start, prefix):
    """Find next action starting with prefix after index start."""
    for j in range(start + 1, len(actions)):
        if actions[j].startswith(prefix):
            return j, times[j]
    return None, None

records = []
for pid in target_ids:
    prows = pl[pl["patient_id"] == pid].sort_values("time")
    actions = prows["action"].tolist()
    times   = prows["time"].tolist()
    triage  = prows["triage"].iloc[0]
    t_short = triage.split("_")[0] if "_" in str(triage) else str(triage)
    n = len(actions)

    durations = {ph: 0.0 for ph in PHASES}
    care_phase_count = 0
    transit_phase_count = 0

    for i, (t, a) in enumerate(zip(times, actions)):

        if a == "WAITING_TRANSPORT:ccp":
            j, tj = _next_action(actions, times, i, "CLOCK_PAUSED:transport")
            if tj is not None:
                durations["Wait transport @ CCP"] += tj - t

        elif a == "WAITING_TRANSPORT:role1":
            j, tj = _next_action(actions, times, i, "CLOCK_PAUSED:transport")
            if tj is not None:
                durations["Wait transport @ R1"] += tj - t

        elif a == "CLOCK_PAUSED:transport":
            # Transit: find next WAITING_CARE or WAITING_TRANSPORT
            for j in range(i + 1, n):
                if actions[j].startswith("WAITING_CARE:") or actions[j].startswith("WAITING_TRANSPORT:"):
                    dt = times[j] - t
                    if transit_phase_count == 0:
                        durations["Transit CCP → R1"] += dt
                    else:
                        durations["Transit R1 → R2"] += dt
                    transit_phase_count += 1
                    break

        elif a.startswith("WAITING_CARE:"):
            care_type = a.split(":", 1)[1]
            j, tj = _next_action(actions, times, i, "CLOCK_PAUSED:care")
            if tj is not None:
                if care_type == "dcs":
                    durations["Wait care @ R2"] += tj - t
                else:
                    durations["Wait care @ R1"] += tj - t

        elif a == "CLOCK_PAUSED:care":
            # In care: find CARE_COMPLETED
            for j in range(i + 1, n):
                if actions[j].startswith("CARE_COMPLETED:"):
                    dt = times[j] - t
                    if care_phase_count == 0:
                        durations["In care @ R1"] += dt
                    else:
                        durations["In DCS care"] += dt
                    care_phase_count += 1
                    break

    records.append({"patient_id": pid, "triage": t_short, **durations})

phase_df = pd.DataFrame(records)
print(f"Computed phase durations for {len(phase_df)} DCS survivors")
phase_df.groupby("triage")[PHASES].mean().round(1)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# PLOT: stacked horizontal bars — typical DCS survivor per triage
# ═══════════════════════════════════════════════════════════════════════════

TRIAGE_ORDER = ["T1", "T2", "T3"]
COLORS = [
    "#e74c3c",  # Wait transport @ CCP
    "#e67e22",  # Transit CCP → R1
    "#f1c40f",  # Wait care @ R1
    "#2ecc71",  # In care @ R1
    "#3498db",  # Wait transport @ R1
    "#9b59b6",  # Transit R1 → R2
    "#1abc9c",  # Wait care @ R2
    "#34495e",  # In DCS care
]

means = phase_df.groupby("triage")[PHASES].mean().reindex(TRIAGE_ORDER).fillna(0)

fig, ax = plt.subplots(figsize=(12, 4))
left = np.zeros(len(TRIAGE_ORDER))

for phase, color in zip(PHASES, COLORS):
    vals = means[phase].values
    bars = ax.barh(TRIAGE_ORDER, vals, left=left, color=color,
                   edgecolor="white", linewidth=0.5, label=phase)
    for j, (v, l) in enumerate(zip(vals, left)):
        if v > means.values.sum(axis=1).max() * 0.04:
            ax.text(l + v / 2, j, f"{v:.0f}", ha="center", va="center",
                    fontsize=9, fontweight="bold", color="white")
    left += vals

ax.set_xlabel("Time (minutes)")
ax.set_title(f"Typical DCS survivor lifetime — {FOLDER_NAME}", fontsize=13)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.15), ncol=4, fontsize=9)
ax.invert_yaxis()
plt.tight_layout()
plt.show()